<a href="https://colab.research.google.com/github/jasonwong-lab/HKU-Practical-Bioinformatics/blob/main/RNA_Seq_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SBMS7100/MMPH6005 Practical Bioinformatics 2025 - RNA-Seq Pipeline

*Prof. Jason Wong*

This lecture aims to demonstrate alignment, quantification, and differential expression analysis of RNA-seq data.


## Package installation and downloads for workshop (~ 5 minutes)

1.   conda (for simple installation of packages)
2.   STAR (for alignment)
3.   subread (for quantification)
4.   samtools (for parsing through data)

**IMPORTANT：Every time you connect to Google Colab, you have to perform these set up steps again.**

In [ ]:
# Set working pathway to your own google drive (~ 1 min)
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Install conda (~ 1 min). There will be a message saying that the session has crashed, but don't worry about this. This is due to the session restarting following conda installation.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Install STAR (~ 2 mins)
!conda install -c bioconda star

In [ ]:
# install samtools, subread (for featureCounts) and fastqc
!conda install -y -c bioconda subread samtools fastqc

In [ ]:
# Install igv-notebook (<1 min)
import sys
print(sys.version, sys.executable)
!{sys.executable} -m pip install -U igv-notebook
import igv_notebook

## Set working directory

By default, the working directory will be My Drive/PB_course

In [ ]:
import os
try:
  os.mkdir("/content/gdrive/My Drive/PB_course")
except FileExistsError:
  print("directory already exist. OK to continue")
os.chdir("/content/gdrive/My Drive/PB_course")

In [ ]:
import os
try:
  os.mkdir("/content/gdrive/My Drive/PB_course/RNA-Seq")
except FileExistsError:
  print("directory already exist. OK to continue")
os.chdir("/content/gdrive/My Drive/PB_course/RNA-Seq")

## Download ready prepared files for analysis.

The RNA-seq data are pairs of normal and tumour samples which have been down sampled to faciliate fast alignment during this workshop.

The reference genome has also been reduced to only contain part of chr6 to reduce the size require on Google Drive.

In [ ]:
# Download Dataset
# Double check that we are in the right directory (~ 30s)
import os

# Change to your desired directory
os.chdir("/content/gdrive/MyDrive/PB_course/RNA-Seq")


if os.path.exists("/content/gdrive/MyDrive/PB_course/RNA-Seq/rna_dataset"):    # check if the folder exist
  print("reference file already exist, OK to continue.")
else:
  !pip install gdown
  !gdown 1_qYYOWNDJ6bpA8rsuAka9SBuqsytGZyU
  # unzip file
  !unzip rna_dataset.zip
  # remove the zip file after extraction
  !rm rna_dataset.zip

!ls -l .

In [ ]:
# Download STAR genome reference files
# Double check that we are in the right directory (~ 30s)
import os

# Change to your desired directory
os.chdir("/content/gdrive/MyDrive/PB_course/RNA-Seq")


if os.path.exists("/content/gdrive/MyDrive/PB_course/RNA-Seq/star_reference"):    # check if the folder exist
  print("reference file already exist, OK to continue.")
else:
  !pip install gdown
  !gdown 1BVCFibsZGCquQoIUJ4B7Fc_zmYrRysGg
  # unzip file
  !unzip star_reference.zip
  # remove the zip file after extraction
  !rm star_reference.zip

#!unzip "/content/gdrive/MyDrive/PB_course/RNA-Seq/star_reference/*.zip" -d "/content/gdrive/MyDrive/PB_course/RNA-Seq/reference/"
!ls -l ./star_reference/


## RNA-seq Alignment and Quantification command line

We will now run through the RNA-seq alignment and quantification pipeline. This will involve the following steps.

1.   fastQC
2.   STAR Alignment
3.   featureCounts
4.   IGV



### 1. fastQC

This is similar to DNA sequencing data we previously analysed.

In [ ]:
# Let's first take a look at the RNA-seq fastq files
%cd /content/gdrive/MyDrive/PB_course/RNA-Seq/rna_dataset
!head -16 2-0019A-1-N-RNA_R1_subset.fastq

In [ ]:
# run fastQC
!fastqc 2-0019A-1-N-RNA_R1_subset.fastq

# Download the html file and check it on your local browser
from google.colab import files
files.download('2-0019A-1-N-RNA_R1_subset_fastqc.html')

### 2. STAR Alignment

STAR has many parameters. We will use default parameters for this example.

In [ ]:
#Let's take a look options in STAR
!STAR --help

In [ ]:
# Make a new directory for STAR output
import os
try:
  os.mkdir("/content/gdrive/My Drive/PB_course/RNA-Seq/STAR")
except FileExistsError:
  print("Clean directory for rerun")
  !rm -rf "/content/gdrive/My Drive/PB_course/RNA-Seq/STAR/"
  os.mkdir("/content/gdrive/My Drive/PB_course/RNA-Seq/STAR")


# List of FASTQ files
fastq_files = [
    "2-0019A-1-N-RNA_R1_subset.fastq",
    "2-0019A-1-N-RNA_R2_subset.fastq",
    "2-0019A-1-T-RNA_R1_subset.fastq",
    "2-0019A-1-T-RNA_R2_subset.fastq"
]

# Run STAR for each pair of FASTQ files
for i in range(0, len(fastq_files), 2):
    r1_file = fastq_files[i]
    r2_file = fastq_files[i + 1]

    # Extract sample name for output prefix
    sample_name = r1_file.split('_')[0]

    # Construct and run the STAR command
    os.system(f"""
    STAR \
      --runThreadN 2 \
      --genomeDir ../star_reference \
      --readFilesIn {r1_file} {r2_file} \
      --outSAMtype BAM SortedByCoordinate \
      --outFileNamePrefix ../STAR/{sample_name}_
    """)

!cat /content/gdrive/MyDrive/PB_course/RNA-Seq/STAR/2-0019A-1-N-RNA_Log.out

###3. featureCounts

FeatureCounts is a tool within the subreads package. It is specifically designed to count the total number of reads overlapping a set of genes defined in a GTF file. There are a range of opinions relating to how to assign reads. Particular attention should be paid to ensure that the correct strand is selected along with how to deal with reads mapping multiple features.

In [ ]:
# Index the BAM files from STAR alignment for faster downstream processing
import os
os.chdir("/content/gdrive/MyDrive/PB_course/RNA-Seq/STAR")

!samtools index 2-0019A-1-T-RNA_Aligned.sortedByCoord.out.bam
!samtools index 2-0019A-1-N-RNA_Aligned.sortedByCoord.out.bam


In [ ]:
# Let's take a look at the annotation.gtf file
!head -20 /content/gdrive/MyDrive/PB_course/RNA-Seq/star_reference/MANE_annotation.gtf

In [ ]:
# Run featureCounts
GTF="/content/gdrive/MyDrive/PB_course/RNA-Seq/star_reference/MANE_annotation.gtf"   # change to your GTF path
OUTDIR="/content/gdrive/MyDrive/PB_course/RNA-Seq/output"
!mkdir -p "$OUTDIR"
os.chdir("/content/gdrive/MyDrive/PB_course/RNA-Seq/output")

!featureCounts \
  -T 2 \
  -t exon -g gene_id \
  -p -B -C \
  -s 0 \
  -a "$GTF" \
  -o "$OUTDIR/featureCounts.txt" \
  ../STAR/*Aligned.sortedByCoord.out.bam

In [ ]:
# Have a look at the raw output files: a) count matrix and b) assignment stats

!head -n 20 /content/gdrive/MyDrive/PB_course/RNA-Seq/output/featureCounts.txt

!column -t /content/gdrive/MyDrive/PB_course/RNA-Seq/output/featureCounts.txt.summary | head -n 20

In [ ]:
# We can also view the output as a table.
import pandas as pd

counts_path = "/content/gdrive/MyDrive/PB_course/RNA-Seq/output/featureCounts.txt"

# Skip comment lines (#) in the header
df = pd.read_csv(counts_path, sep="\t", comment="#")

# Explore structure
df.columns.tolist()
df.shape
df.head()

In [ ]:
# Most of the  genes in the annotation file don't have reads in our samples. We can view just the genes that have reads:
import pandas as pd

counts_path = "/content/gdrive/MyDrive/PB_course/RNA-Seq/output/featureCounts.txt"

# Load (skip comment lines)
df = pd.read_csv(counts_path, sep="\t", comment="#")

# Extract count columns (everything from col7 onward)
count_cols = df.columns[6:]
counts_only = df[count_cols]

# Keep only rows with at least one nonzero count
df_nonzero = df[(counts_only > 0).any(axis=1)]

print("Original genes:", df.shape[0])
print("Genes with counts:", df_nonzero.shape[0])

# Inspect a few
df_nonzero.head()

###4. IGV



In [ ]:
# Load track from local paths
import os
import igv_notebook
os.chdir("/content/gdrive/MyDrive/PB_course/RNA-Seq/STAR")

igv_notebook.init()

b = igv_notebook.Browser(
    {
        "genome": "hg38",
        "locus": "DDX39B"
    }
)

# We're loading both the tumour bam file from sample 2-0019A
b.load_track(
    {
        "name": "RNA-Seq",
        "path": "./2-0019A-1-T-RNA_Aligned.sortedByCoord.out.bam",
        "indexPath": "./2-0019A-1-T-RNA_Aligned.sortedByCoord.out.bam.bai",
        "format": "bam",
        "type": "alignment"
    })